In [ ]:
import os
import json
from datetime import datetime

# 1. 부정적 요인 분석 함수
def check_negative_sentiment(text):
    negative_keywords = ["별로", "최악", "불친절", "맛없", "느려", "비싸", "웨이팅", "불쾌", "시끄러", "좁아"]
    found_negatives = [word for word in negative_keywords if word in text]
    return list(set(found_negatives))

# 2. 식당 분석 함수 (개별 파일 처리)
def analyze_restaurant_data(file_path):
    # --- [수정] 파일 이름에서 식당 이름 추출 ---
    filename = os.path.basename(file_path) 
    # 예: '연남황제소갈비집_2062344030.json' -> 이름: '연남황제소갈비집'
    restaurant_name = filename.rsplit('_', 1)[0] 
    # ----------------------------------------

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # 만약 JSON 내부에도 'restaurant_name'이 있다면 그것을 우선 사용
    display_name = data.get('restaurant_name', restaurant_name)
    
    reviews = data.get("reviews", [])
    if not reviews: return

    total_volume = len(reviews)
    recent_reviews = 0
    now = datetime(2026, 6, 10)
    
    tag_counts = {"강아지동반": 0, "가성비/무한리필": 0, "술자리/회식": 0, "친절/서비스": 0, "야근/심야": 0, "주차편리": 0, "맛있음": 0}
    negative_summary = {}
    unique_reviews = {r['id']: r for r in reviews}.values()

    for r in unique_reviews:
        body = r.get("body", "")
        try:
            visited_dt = datetime.strptime(r['visited_date'], "%Y.%m.%d.")
            if (now - visited_dt).days <= 90: recent_reviews += 1
        except: pass
        
        # 키워드 맵을 훨씬 풍부하게 확장하세요
        keyword_map = {
            "친절/서비스": ["친절", "서비스", "사장님", "직원", "상냥", "웃음", "친절하시"],
            "술자리/회식": ["술", "2차", "회식", "안주", "소주", "맥주", "술집", "분위기"],
            "가성비/무한리필": ["무한리필", "가성비", "저렴", "싼", "양 많", "배불"],
            "맛있음": ["맛있", "존맛", "꿀맛", "최고", "환상", "입에서 녹"] # '맛있음' 페르소나 추가
        }
        for tag, keywords in keyword_map.items():
            if any(k in body for k in keywords): tag_counts[tag] += 1
        
        for neg in check_negative_sentiment(body):
            negative_summary[neg] = negative_summary.get(neg, 0) + 1

    print(f"\n=== [분석 리포트: {display_name}] ===")
    print(f"총 {total_volume}건 (최근 90일: {recent_reviews}건)")
    print("페르소나:", {k: v for k, v in tag_counts.items() if v > 0})
    if negative_summary: print("주의 키워드:", negative_summary)

# 3. 메인 실행 함수 (경로 자동 설정)
def run_batch_analysis():
    # 현재 작업 폴더(os.getcwd())로부터 상대 경로를 찾아나감
    # code/magicwizard 내에 있다면 상위 2단계가 프로젝트 루트임
    root_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
    target_directory = os.path.join(root_dir, "data/mapo_district")
    
    if not os.path.exists(target_directory):
        print(f"경로를 찾을 수 없습니다: {target_directory}")
        return

    files = [f for f in os.listdir(target_directory) if f.endswith('.json')]
    print(f"분석 대상 식당 {len(files)}개 발견!")
    
    for filename in files:
        analyze_restaurant_data(os.path.join(target_directory, filename))

if __name__ == "__main__":
    run_batch_analysis()

분석 대상 식당 17개 발견!

=== [분석 리포트: 연남황제소갈비집] ===
총 100건 (최근 90일: 50건)
페르소나: {'가성비/무한리필': 4, '술자리/회식': 9, '친절/서비스': 25, '맛있음': 38}
주의 키워드: {'별로': 2, '맛없': 1}

=== [분석 리포트: 인생제육_망원점] ===
총 1건 (최근 90일: 1건)
페르소나: {'친절/서비스': 1}

=== [분석 리포트: 돼지방앗간] ===
총 150건 (최근 90일: 50건)
페르소나: {'가성비/무한리필': 2, '술자리/회식': 3, '친절/서비스': 12, '맛있음': 38}
주의 키워드: {'웨이팅': 1}

=== [분석 리포트: 고기꾼김춘배_공덕직영점] ===
총 3700건 (최근 90일: 50건)
페르소나: {'술자리/회식': 9, '친절/서비스': 5, '맛있음': 36}
주의 키워드: {'웨이팅': 1, '맛없': 1}

=== [분석 리포트: 원조부안집_마포직영점] ===
총 3800건 (최근 90일: 50건)
페르소나: {'가성비/무한리필': 3, '술자리/회식': 5, '친절/서비스': 21, '맛있음': 38}
주의 키워드: {'웨이팅': 2, '별로': 1}

=== [분석 리포트: 대장육_마포점] ===
총 100건 (최근 90일: 50건)
페르소나: {'술자리/회식': 1, '친절/서비스': 12, '맛있음': 16}

=== [분석 리포트: 홍대_육지] ===
총 8900건 (최근 90일: 50건)
페르소나: {'가성비/무한리필': 1, '술자리/회식': 5, '친절/서비스': 13, '맛있음': 41}
주의 키워드: {'별로': 1, '웨이팅': 1}

=== [분석 리포트: 로코스비비큐_상수점] ===
총 28건 (최근 90일: 28건)
페르소나: {'가성비/무한리필': 2, '술자리/회식': 8, '친절/서비스': 11, '맛있음': 18}

=== [분석 리포트: 성산제줏간_까치산점] ===
총 200건 (최근 90일: 50건)
